In [ ]:
# ============================================================
# MULTIMODAL FUSION MODEL
# Cognitive Features + MRI Probability Outputs
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# ============================================================
# RELIABLE GOOGLE DRIVE MOUNT
# ============================================================

import os
import shutil
from google.colab import drive

MOUNT_POINT = "/content/drive"

# Case 1: Drive is already properly mounted
if os.path.ismount(MOUNT_POINT):
    print("Google Drive is already mounted.")

# Case 2: The folder exists but is only a stale, unmounted folder
else:
    if os.path.exists(MOUNT_POINT):
        print("Removing stale /content/drive folder...")
        raise RuntimeError("Choose a fresh Colab runtime before mounting Drive.")

    # Create a clean mount point
    os.makedirs(MOUNT_POINT, exist_ok=True)

    # Mount Google Drive
    drive.mount(MOUNT_POINT)

# Final verification
if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive is ready.")
    print("Mounted status:", os.path.ismount(MOUNT_POINT))
else:
    raise RuntimeError("Google Drive mounting failed.")

In [ ]:
# ============================================================
# 1. Load cognitive dataset and MRI files
# ============================================================

mri_folder = "/content/drive/MyDrive/ADNI_Project/MRI_Images/Processed_MRI_160"

final_cognitive = pd.read_csv(
    "/content/drive/MyDrive/ADNI_Project/Cognitive_Data/final_cognitive_702.csv"
)

files = sorted([f for f in os.listdir(mri_folder) if f.endswith(".npy")])

label_map = dict(zip(final_cognitive["RID"], final_cognitive["DIAGNOSIS"]))

mri_records = []

for file in files:
    rid = int(re.search(r"RID_(.*?)_IMG", file).group(1))

    if rid in label_map:
        diagnosis = label_map[rid]

        mri_records.append({
            "RID": rid,
            "MRI_FILE": file,
            "MRI_PATH": os.path.join(mri_folder, file),
            "DIAGNOSIS": diagnosis
        })

mri_dataset = pd.DataFrame(mri_records)
mri_dataset["LABEL"] = mri_dataset["DIAGNOSIS"].astype(int) - 1

print("MRI dataset:", mri_dataset.shape)
print("Cognitive dataset:", final_cognitive.shape)

In [ ]:
# ============================================================
# 2. LOAD THE EXACT OFFICIAL MRI DATA SPLITS
# ============================================================

MRI_RESULTS_FOLDER = (
    "/content/drive/MyDrive/ADNI_Project/"
    "Dissertation_Results/MRI_Results_Official"
)

train_df = pd.read_csv(
    os.path.join(
        MRI_RESULTS_FOLDER,
        "MRI_training_split.csv"
    )
)

val_df = pd.read_csv(
    os.path.join(
        MRI_RESULTS_FOLDER,
        "MRI_validation_split.csv"
    )
)

test_df = pd.read_csv(
    os.path.join(
        MRI_RESULTS_FOLDER,
        "MRI_test_split.csv"
    )
)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

print("\nTraining classes:")
print(train_df["LABEL"].value_counts().sort_index())

print("\nValidation classes:")
print(val_df["LABEL"].value_counts().sort_index())

print("\nTesting classes:")
print(test_df["LABEL"].value_counts().sort_index())

In [ ]:
# ============================================================
# 3. LOAD THE NEW OFFICIAL MRI MODEL
# ============================================================

mri_model_path = (
    "/content/drive/MyDrive/ADNI_Project/"
    "Interface_Models/"
    "best_mri_3slice_mobilenet_model.keras"
)

if not os.path.isfile(mri_model_path):
    raise FileNotFoundError(
        "The new official MRI model was not found:\n"
        f"{mri_model_path}"
    )

mri_model = tf.keras.models.load_model(
    mri_model_path
)

print("New official MRI model loaded successfully.")
print("Input shape:", mri_model.input_shape)
print("Output shape:", mri_model.output_shape)


In [ ]:
# ============================================================
# 4. CREATE THREE-SLICE MRI DATASET FOR PREDICTION
# MUST MATCH THE OFFICIAL MRI TRAINING PREPROCESSING
# ============================================================

IMG_SIZE = 160
BATCH_SIZE = 16

SLICE_1 = 60
SLICE_2 = 80
SLICE_3 = 100

EXPECTED_VOLUME_SHAPE = (160, 160, 160)


def load_three_mri_slices(path, label):

    path = path.numpy().decode("utf-8")

    volume = np.load(
        path,
        allow_pickle=False
    ).astype(np.float32)

    if volume.shape != EXPECTED_VOLUME_SHAPE:
        raise ValueError(
            f"Unexpected MRI shape {volume.shape} for {path}. "
            f"Expected {EXPECTED_VOLUME_SHAPE}."
        )

    slice_1 = volume[:, :, SLICE_1]
    slice_2 = volume[:, :, SLICE_2]
    slice_3 = volume[:, :, SLICE_3]

    image = np.stack(
        [slice_1, slice_2, slice_3],
        axis=-1
    )

    image = np.clip(
        image,
        0.0,
        1.0
    )

    # This must match the official MRI model training
    image = preprocess_input(
        image * 255.0
    )

    return (
        image.astype(np.float32),
        np.int64(label)
    )


def tf_load_three_mri_slices(path, label):

    image, label = tf.py_function(
        func=load_three_mri_slices,
        inp=[path, label],
        Tout=[tf.float32, tf.int64]
    )

    image.set_shape(
        (IMG_SIZE, IMG_SIZE, 3)
    )

    label.set_shape([])

    return image, label


def create_mri_prediction_dataset(df):

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            df["MRI_PATH"].values,
            df["LABEL"].values
        )
    )

    dataset = (
        dataset
        .map(
            tf_load_three_mri_slices,
            num_parallel_calls=1
        )
        .batch(BATCH_SIZE)
        .prefetch(1)
    )

    return dataset


print("MRI prediction loader created successfully.")

In [ ]:
# ============================================================
# 5. GENERATE NEW MRI PROBABILITY FEATURES
# ============================================================

def get_mri_probabilities(df, split_name):

    print(f"\nGenerating MRI probabilities for {split_name}...")

    dataset = create_mri_prediction_dataset(df)

    probabilities = mri_model.predict(
        dataset,
        verbose=1
    )

    if probabilities.shape != (len(df), 3):
        raise ValueError(
            f"Unexpected prediction shape: {probabilities.shape}"
        )

    probability_df = pd.DataFrame({
        "RID": df["RID"].astype(int).values,
        "MRI_PROB_CN": probabilities[:, 0],
        "MRI_PROB_MCI": probabilities[:, 1],
        "MRI_PROB_AD": probabilities[:, 2],
        "LABEL": df["LABEL"].astype(int).values
    })

    return probability_df


train_mri_probs = get_mri_probabilities(
    train_df,
    "training set"
)

val_mri_probs = get_mri_probabilities(
    val_df,
    "validation set"
)

test_mri_probs = get_mri_probabilities(
    test_df,
    "test set"
)

print("\nNew MRI probability features created successfully.")

display(train_mri_probs.head())

In [ ]:
# ============================================================
# 6. PREPARE COGNITIVE FEATURES
# ============================================================

cognitive_data = final_cognitive.copy()

cognitive_data["LABEL"] = (
    cognitive_data["DIAGNOSIS"].astype(int) - 1
)

drop_columns = [
    "RID",
    "VISCODE",
    "DIAGNOSIS",
    "LABEL"
]

cognitive_feature_columns = [
    column
    for column in cognitive_data.columns
    if column not in drop_columns
]

print("Number of cognitive features:",
      len(cognitive_feature_columns))

print("\nCognitive features:")
print(cognitive_feature_columns)

In [ ]:
# ============================================================
# 7. MERGE COGNITIVE FEATURES WITH MRI PROBABILITIES
# ============================================================

MRI_PROBABILITY_COLUMNS = [
    "MRI_PROB_CN",
    "MRI_PROB_MCI",
    "MRI_PROB_AD"
]

FUSION_FEATURE_COLUMNS = (
    cognitive_feature_columns
    + MRI_PROBABILITY_COLUMNS
)


def create_fusion_dataset(mri_probability_df):

    fusion_df = mri_probability_df.merge(
        cognitive_data,
        on="RID",
        how="inner",
        suffixes=("_MRI", "_COG")
    )

    X = fusion_df[
        FUSION_FEATURE_COLUMNS
    ].copy()

    y = fusion_df[
        "LABEL_MRI"
    ].astype(int).copy()

    return X, y, fusion_df


X_train_fusion, y_train_fusion, train_fusion_df = (
    create_fusion_dataset(train_mri_probs)
)

X_val_fusion, y_val_fusion, val_fusion_df = (
    create_fusion_dataset(val_mri_probs)
)

X_test_fusion, y_test_fusion, test_fusion_df = (
    create_fusion_dataset(test_mri_probs)
)

print("Fusion training data:", X_train_fusion.shape)
print("Fusion validation data:", X_val_fusion.shape)
print("Fusion testing data:", X_test_fusion.shape)

assert len(X_train_fusion) == 448
assert len(X_val_fusion) == 113
assert len(X_test_fusion) == 141

assert list(X_train_fusion.columns) == FUSION_FEATURE_COLUMNS

print("\nFusion datasets created successfully.")

In [ ]:
# ============================================================
# 8. TRAIN THE UPDATED FUSION MODEL
# ============================================================

fusion_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

fusion_model.fit(
    X_train_fusion,
    y_train_fusion
)

print("Updated fusion model trained successfully.")

In [ ]:
# ============================================================
# 9. EVALUATE VALIDATION SET
# ============================================================

val_preds = fusion_model.predict(
    X_val_fusion
)

val_accuracy = accuracy_score(
    y_val_fusion,
    val_preds
)

print(
    f"Validation accuracy: "
    f"{val_accuracy * 100:.2f}%"
)

print("\nValidation confusion matrix:")
print(
    confusion_matrix(
        y_val_fusion,
        val_preds
    )
)

print("\nValidation classification report:")
print(
    classification_report(
        y_val_fusion,
        val_preds,
        target_names=["CN", "MCI", "AD"],
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 10. EVALUATE INDEPENDENT TEST SET
# ============================================================

test_preds = fusion_model.predict(
    X_test_fusion
)

test_accuracy = accuracy_score(
    y_test_fusion,
    test_preds
)

test_confusion_matrix = confusion_matrix(
    y_test_fusion,
    test_preds
)

test_report = classification_report(
    y_test_fusion,
    test_preds,
    target_names=["CN", "MCI", "AD"],
    output_dict=True,
    zero_division=0
)

print(
    f"Test accuracy: "
    f"{test_accuracy * 100:.2f}%"
)

print("\nTest confusion matrix:")
print(test_confusion_matrix)

print("\nTest classification report:")
print(
    classification_report(
        y_test_fusion,
        test_preds,
        target_names=["CN", "MCI", "AD"],
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 11. SAVE THE UPDATED FUSION MODEL
# ============================================================
import joblib
UPDATED_FUSION_RESULTS_FOLDER = (
    "/content/drive/MyDrive/ADNI_Project/"
    "Dissertation_Results/Fusion_Results_Official"
)

os.makedirs(
    UPDATED_FUSION_RESULTS_FOLDER,
    exist_ok=True
)

UPDATED_FUSION_MODEL_PATH = os.path.join(
    UPDATED_FUSION_RESULTS_FOLDER,
    "official_fusion_random_forest.joblib"
)

joblib.dump(
    fusion_model,
    UPDATED_FUSION_MODEL_PATH
)

FUSION_FEATURES_PATH = os.path.join(
    UPDATED_FUSION_RESULTS_FOLDER,
    "official_fusion_feature_columns.joblib"
)

joblib.dump(
    FUSION_FEATURE_COLUMNS,
    FUSION_FEATURES_PATH
)

print("Updated fusion model saved to:")
print(UPDATED_FUSION_MODEL_PATH)

print("\nFusion feature columns saved to:")
print(FUSION_FEATURES_PATH)

In [ ]:
# ============================================================
# 12. SAVE UPDATED FUSION RESULTS
# ============================================================

report_df = pd.DataFrame(
    test_report
).transpose()

report_path = os.path.join(
    UPDATED_FUSION_RESULTS_FOLDER,
    "official_fusion_classification_report.csv"
)

report_df.to_csv(report_path)

confusion_path = os.path.join(
    UPDATED_FUSION_RESULTS_FOLDER,
    "official_fusion_confusion_matrix.csv"
)

pd.DataFrame(
    test_confusion_matrix,
    index=["Actual_CN", "Actual_MCI", "Actual_AD"],
    columns=["Predicted_CN", "Predicted_MCI", "Predicted_AD"]
).to_csv(confusion_path)

summary_df = pd.DataFrame({
    "Metric": [
        "Validation accuracy",
        "Test accuracy",
        "Training participants",
        "Validation participants",
        "Test participants",
        "Number of cognitive features",
        "Number of MRI probability features",
        "Total fusion features"
    ],
    "Value": [
        val_accuracy,
        test_accuracy,
        len(X_train_fusion),
        len(X_val_fusion),
        len(X_test_fusion),
        len(cognitive_feature_columns),
        len(MRI_PROBABILITY_COLUMNS),
        len(FUSION_FEATURE_COLUMNS)
    ]
})

summary_path = os.path.join(
    UPDATED_FUSION_RESULTS_FOLDER,
    "official_fusion_results_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

display(summary_df)

print("\nAll updated fusion results saved in:")
print(UPDATED_FUSION_RESULTS_FOLDER)

In [ ]:
# ============================================================
# 13. UPDATED FUSION FEATURE IMPORTANCE
# ============================================================

feature_importance_df = pd.DataFrame({
    "Feature": FUSION_FEATURE_COLUMNS,
    "Importance": fusion_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

display(feature_importance_df)

feature_importance_path = os.path.join(
    UPDATED_FUSION_RESULTS_FOLDER,
    "official_fusion_feature_importance.csv"
)

feature_importance_df.to_csv(
    feature_importance_path,
    index=False
)

print("Feature importance saved to:")
print(feature_importance_path)

In [ ]:
# ============================================================
# SAVE THE MULTIMODAL FUSION RANDOM FOREST MODEL
# ============================================================

import os
import joblib

# Same folder used for all interface models
MODEL_FOLDER = "/content/drive/MyDrive/ADNI_Project/Interface_Models"

os.makedirs(MODEL_FOLDER, exist_ok=True)

# Save the trained fusion model
fusion_model_path = os.path.join(
    MODEL_FOLDER,
    "fusion_random_forest.joblib"
)

joblib.dump(fusion_model, fusion_model_path)

print("Fusion model saved successfully.")
print("Location:", fusion_model_path)

In [ ]:
# ============================================================
# SAVE FUSION FEATURE COLUMN ORDER
# ============================================================

import os
import joblib

MODEL_FOLDER = "/content/drive/MyDrive/ADNI_Project/Interface_Models"

os.makedirs(MODEL_FOLDER, exist_ok=True)

fusion_feature_columns = X_train_fusion.columns.tolist()

fusion_features_path = os.path.join(
    MODEL_FOLDER,
    "fusion_feature_columns.joblib"
)

joblib.dump(
    fusion_feature_columns,
    fusion_features_path
)

print("Fusion feature columns saved successfully.")
print("Number of features:", len(fusion_feature_columns))
print("\nFeature order:")

for feature in fusion_feature_columns:
    print("-", feature)

In [ ]:
import os

files = os.listdir("/content/drive/MyDrive/ADNI_Project/Interface_Models")

print("Saved files:")

for f in files:
    print("-", f)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# ---------------------------------------
# Official fusion-model test confusion matrix
# Rows = actual classes
# Columns = predicted classes
# ---------------------------------------

test_cm = np.array([
    [39,  0,  0],   # Actual CN
    [ 0, 68,  2],   # Actual MCI
    [ 0, 10, 22]    # Actual AD
])

class_names = ["CN", "MCI", "AD"]

# ---------------------------------------
# Create figure
# ---------------------------------------

fig, ax = plt.subplots(figsize=(7, 6))

display = ConfusionMatrixDisplay(
    confusion_matrix=test_cm,
    display_labels=class_names
)

display.plot(
    cmap="Oranges",
    values_format="d",
    colorbar=False,
    ax=ax
)

ax.set_title(
    "Multimodal Fusion Model: Test Confusion Matrix",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Predicted Class", fontsize=12)
ax.set_ylabel("Actual Class", fontsize=12)

ax.tick_params(axis="both", labelsize=11)

plt.tight_layout()

# ---------------------------------------
# Save figure
# ---------------------------------------

OUTPUT_DIR = (
    "/content/drive/MyDrive/ADNI_Project/"
    "Dissertation_Results/Fusion_Results_Official"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = os.path.join(
    OUTPUT_DIR,
    "fusion_test_confusion_matrix.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Confusion matrix saved to:")
print(output_path)

In [ ]:
# ============================================================
# PROFESSIONAL FEATURE IMPORTANCE PLOT
# MULTIMODAL FUSION MODEL
# ============================================================

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import os

# ------------------------------------------------------------
# Create dataframe
# ------------------------------------------------------------

feature_importance = pd.DataFrame({
    "Feature": X_train_fusion.columns,
    "Importance": fusion_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Top N features
# ------------------------------------------------------------

TOP_N = 15

plot_df = feature_importance.head(TOP_N).copy()

# Reverse for horizontal plotting
plot_df = plot_df.iloc[::-1]

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 8))

# Purple → Green gradient (same style as cognitive model)

colors = plt.cm.viridis(np.linspace(0.05, 0.75, len(plot_df)))

bars = ax.barh(
    plot_df["Feature"],
    plot_df["Importance"],
    color=colors,
    edgecolor="white",
    linewidth=1.5
)

# ------------------------------------------------------------
# Value labels
# ------------------------------------------------------------

for bar in bars:
    width = bar.get_width()

    ax.text(
        width + 0.004,
        bar.get_y() + bar.get_height()/2,
        f"{width:.4f}",
        va="center",
        ha="left",
        fontsize=12,
        fontweight="bold",
        color="#273142"
    )

# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

ax.set_title(
    "Multimodal Fusion Feature Importance",
    fontsize=24,
    fontweight="bold",
    pad=18
)

ax.set_xlabel(
    "Relative Contribution",
    fontsize=18,
    fontweight="bold",
    labelpad=18
)

ax.set_ylabel(
    "Fusion Features",
    fontsize=18,
    fontweight="bold",
    labelpad=18
)

ax.tick_params(axis='both', labelsize=13)

ax.xaxis.grid(
    True,
    linestyle="--",
    alpha=0.35
)

ax.set_axisbelow(True)

# Remove top/right borders
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ------------------------------------------------------------
# Highlight top 3 features
# ------------------------------------------------------------

top3 = ", ".join(feature_importance.head(3)["Feature"])

plt.figtext(
    0.5,
    0.02,
    f"Most influential features: {top3}",
    ha="center",
    fontsize=15,
    fontweight="bold",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="#eef9f0",
        edgecolor="#7ecf9b",
        linewidth=1.5
    )
)

plt.tight_layout(rect=[0,0.05,1,1])

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

OUTPUT_DIR = "/content/drive/MyDrive/ADNI_Project/Dissertation_Results/Fusion_Results_Official"

os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "fusion_feature_importance_professional.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

# ---------------------------------------------------------
# Generate Classification Report
# ---------------------------------------------------------

report = classification_report(
     y_test_fusion,
    test_preds,
    target_names=["CN", "MCI", "AD"],
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()

# Round values to 3 decimal places
report_df = report_df.round(3)

report_df = (
    pd.DataFrame(report)
    .transpose()
    .round(3)
)

print(report_df)

# ---------------------------------------------------------
# Save to CSV
# ---------------------------------------------------------

OUTPUT_DIR = "/content/drive/MyDrive/ADNI_Project/Dissertation_Results/Fusion_Results_Official"

report_df.to_csv(
    f"{OUTPUT_DIR}/fusion_classification_report.csv"
)

print("Classification report saved successfully.")

In [ ]:
# ============================================================
# VISUAL 1: TEST ACCURACY COMPARISON
# MRI vs Cognitive vs Multimodal Fusion
# ============================================================

import os
import matplotlib.pyplot as plt
import pandas as pd

# ------------------------------------------------------------
# Final Test Accuracies (%)
# ------------------------------------------------------------

accuracy_data = pd.DataFrame({
    "Model": [
        "MRI-only",
        "Cognitive-only",
        "Multimodal Fusion"
    ],
    "Test Accuracy (%)": [
        41.84,
        94.33,
        91.49
    ]
})

print(accuracy_data)

# ------------------------------------------------------------
# Create output folder
# ------------------------------------------------------------

output_folder = "Dissertation_Results/Model_Comparison"
os.makedirs(output_folder, exist_ok=True)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    accuracy_data["Model"],
    accuracy_data["Test Accuracy (%)"],
    width=0.6,
    edgecolor="black",
    linewidth=1.2
)

# Value labels
for bar, value in zip(bars, accuracy_data["Test Accuracy (%)"]):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 1.2,
        f"{value:.2f}%",
        ha="center",
        fontsize=12,
        fontweight="bold"
    )

# Formatting
ax.set_title(
    "Comparison of Test Accuracy Across Classification Models",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "Model",
    fontsize=12,
    fontweight="bold"
)

ax.set_ylabel(
    "Test Accuracy (%)",
    fontsize=12,
    fontweight="bold"
)

ax.set_ylim(0, 105)

ax.grid(axis="y", linestyle="--", alpha=0.4)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.xticks(fontsize=11, fontweight="bold")
plt.yticks(fontsize=10)

plt.tight_layout()

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

plt.savefig(
    os.path.join(output_folder, "Accuracy_Comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(output_folder, "Accuracy_Comparison.pdf"),
    bbox_inches="tight"
)

plt.show()

print("Accuracy comparison figure saved successfully.")

In [ ]:
# ============================================================
# VISUAL 2: OVERALL MODEL PERFORMANCE COMPARISON
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# Final evaluation metrics
# ------------------------------------------------------------

metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],

    "MRI-only": [
        41.84,
        30.30,
        31.90,
        29.90
    ],

    "Cognitive-only": [
        94.33,
        94.00,
        94.00,
        94.00
    ],

    "Multimodal Fusion": [
        91.49,
        92.90,
        88.60,
        90.20
    ]
})

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

output_folder = "Dissertation_Results/Model_Comparison"
os.makedirs(output_folder, exist_ok=True)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

x = np.arange(len(metrics["Metric"]))

width = 0.25

fig, ax = plt.subplots(figsize=(12,6))

colors = [
    "#B0B0B0",   # MRI - Grey
    "#4C78A8",   # Cognitive - Blue
    "#72B7B2"    # Fusion - Teal
]

bars1 = ax.bar(
    x-width,
    metrics["MRI-only"],
    width,
    label="MRI-only",
    color=colors[0],
    edgecolor="black"
)

bars2 = ax.bar(
    x,
    metrics["Cognitive-only"],
    width,
    label="Cognitive-only",
    color=colors[1],
    edgecolor="black"
)

bars3 = ax.bar(
    x+width,
    metrics["Multimodal Fusion"],
    width,
    label="Multimodal Fusion",
    color=colors[2],
    edgecolor="black"
)

# ------------------------------------------------------------
# Add labels
# ------------------------------------------------------------

def add_labels(bars):

    for bar in bars:

        h = bar.get_height()

        ax.text(
            bar.get_x()+bar.get_width()/2,
            h+1,
            f"{h:.1f}",
            ha="center",
            fontsize=10
        )

add_labels(bars1)
add_labels(bars2)
add_labels(bars3)

# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

ax.set_xticks(x)

ax.set_xticklabels(
    metrics["Metric"],
    fontsize=11,
    fontweight="bold"
)

ax.set_ylabel(
    "Performance (%)",
    fontsize=12,
    fontweight="bold"
)

ax.set_ylim(0,105)

ax.set_title(
    "Comparison of Overall Classification Performance",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

ax.legend()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

plt.savefig(
    os.path.join(output_folder,
                 "Overall_Performance_Comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(output_folder,
                 "Overall_Performance_Comparison.pdf"),
    bbox_inches="tight"
)

plt.show()

print("Overall performance comparison saved successfully.")

In [ ]:
# ============================================================
# CLASS-WISE F1-SCORE HEATMAP
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# F1-score data
# ------------------------------------------------------------

f1_scores = pd.DataFrame(
    {
        "CN": [0.119, 1.000, 1.000],
        "MCI": [0.614, 0.940, 0.919],
        "AD": [0.163, 0.880, 0.786]
    },
    index=[
        "MRI-only",
        "Cognitive-only",
        "Multimodal Fusion"
    ]
)

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

output_folder = "Dissertation_Results/Model_Comparison"
os.makedirs(output_folder, exist_ok=True)

# ------------------------------------------------------------
# Create heatmap
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8,5))

heatmap = ax.imshow(
    f1_scores,
    cmap="YlOrRd",
    aspect="auto",
    vmin=0,
    vmax=1
)

# ------------------------------------------------------------
# Tick labels
# ------------------------------------------------------------

ax.set_xticks(np.arange(len(f1_scores.columns)))
ax.set_yticks(np.arange(len(f1_scores.index)))

ax.set_xticklabels(
    f1_scores.columns,
    fontsize=12,
    fontweight="bold"
)

ax.set_yticklabels(
    f1_scores.index,
    fontsize=11,
    fontweight="bold"
)

# ------------------------------------------------------------
# Add values inside cells
# ------------------------------------------------------------

for i in range(f1_scores.shape[0]):
    for j in range(f1_scores.shape[1]):
        value = f1_scores.iloc[i, j]

        ax.text(
            j,
            i,
            f"{value:.3f}",
            ha="center",
            va="center",
            color="black",
            fontsize=11,
            fontweight="bold"
        )

# ------------------------------------------------------------
# Colour bar
# ------------------------------------------------------------

cbar = plt.colorbar(heatmap)
cbar.set_label(
    "F1-score",
    fontsize=11,
    fontweight="bold"
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

ax.set_xlabel(
    "Diagnostic Class",
    fontsize=12,
    fontweight="bold"
)

ax.set_ylabel(
    "Model",
    fontsize=12,
    fontweight="bold"
)

ax.set_title(
    "Class-wise F1-score Heatmap",
    fontsize=15,
    fontweight="bold",
    pad=15
)

plt.tight_layout()

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

plt.savefig(
    os.path.join(output_folder,
                 "Classwise_F1_Heatmap.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(output_folder,
                 "Classwise_F1_Heatmap.pdf"),
    bbox_inches="tight"
)

plt.show()

print("Heatmap saved successfully.")